# 머신러닝 기초 2장. 배깅과 랜덤 포레스트

사회약학 연구방법 노트 '머신러닝 기초' 2장을 따라 하는 노트북입니다. 0장과 같은 가상 청구자료와 분할로 배깅과 랜덤 포레스트를 만들고, 조절값을 격자 탐색, 무작위 탐색, 단계적 탐색, Optuna, HyperOpt로 튜닝한 뒤 변수 중요도와 부분 의존 그림으로 결과를 정리합니다. 셀을 위에서부터 차례로 실행하세요. 탐색 셀은 Colab에서 몇 분씩 걸립니다. 자료는 사이트가 만든 가상 자료이며 실제 환자 자료가 아닙니다.


## 준비

0장의 자료 불러오기, 분할, 전처리 함수(make_pipe), 5겹 분할기(skf)와 1장의 나무용 전처리 함수(make_tree_pipe)를 한 셀에 모았습니다.

### 셀 1. 0장과 1장의 준비를 한 번에 (자료, 분할, 파이프라인, skf)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import (train_test_split,
                                     StratifiedKFold, cross_val_score)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score

BASE = "https://socialp-ajou.tecentriq12.workers.dev/data/"
df = pd.read_csv(BASE + "ml_claims.csv")
y = df["admit_2023"]
X = df.drop(columns=["id", "admit_2023", "cost_2023"])
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=2026)

cat_cols = ["insurance", "region", "smoking", "alcohol"]
num_cols = [c for c in X.columns if c not in cat_cols]

def make_pipe(model, num=num_cols, cat=cat_cols):
    """0장 다 절과 같은 전처리(결측 대체, 표준화, 원-핫) + 모형"""
    num_steps = Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale", StandardScaler())])
    cat_steps = Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])
    prep = ColumnTransformer([("num", num_steps, num),
                              ("cat", cat_steps, cat)])
    return Pipeline([("prep", prep), ("model", model)])

def make_tree_pipe(model, num=num_cols, cat=cat_cols):
    """1장 가 절과 같은 나무용: 결측 대체와 원-핫만 (표준화 없음)"""
    cat_steps = Pipeline([
        ("impute", SimpleImputer(strategy="constant",
                                 fill_value="missing")),
        ("onehot", OneHotEncoder(handle_unknown="ignore",
                                 sparse_output=False))])
    prep = ColumnTransformer(
        [("num", SimpleImputer(strategy="median"), num),
         ("cat", cat_steps, cat)],
        verbose_feature_names_out=False)
    return Pipeline([("prep", prep), ("model", model)])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=2026)
print(X_train.shape, X_test.shape, y_train.sum(), y_test.sum())

## 가. 여러 나무를 평균 내는 이유

### 셀 2. 부트스트랩으로 나무 100그루를 키워 평균 내기

In [ ]:
from sklearn.tree import DecisionTreeClassifier

prep = make_tree_pipe(None).named_steps["prep"]  # 전처리만
Z_train = prep.fit_transform(X_train)          # 11250 × 48 배열
Z_test = prep.transform(X_test)                # 3750 × 48
y_arr = y_train.to_numpy()

rng = np.random.default_rng(2026)
n = len(Z_train)
probs = []                                     # 나무마다 시험 자료 확률
for b in range(100):
    idx = rng.integers(0, n, size=n)           # ① 복원추출로 한 벌
    tree = DecisionTreeClassifier(min_samples_leaf=20,
                                  random_state=b)
    tree.fit(Z_train[idx], y_arr[idx])         # ② 나무 한 그루
    probs.append(tree.predict_proba(Z_test)[:, 1])
probs = np.array(probs)                        # (100, 3750)
p_avg = probs.mean(axis=0)                     # ③ 사람마다 평균

print(probs.shape, " people in the last bag:", len(np.unique(idx)))
who = np.argsort(p_avg)[[1874, 3599, 3749]]    # 중간, 상위 4%, 최고
print(probs[:5, who].round(3))                 # 나무 5그루 × 3명
print("SD over trees:", probs[:, who].std(axis=0).round(3))
print("average      :", p_avg[who].round(3))
aucs = [roc_auc_score(y_test, p) for p in probs]
print("AUC, one tree:", round(min(aucs), 3), "to",
      round(max(aucs), 3))
print("AUC, average :", round(roc_auc_score(y_test, p_avg), 3))

### 셀 3. 나무끼리 얼마나 닮았나 (배깅과 랜덤 포레스트)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

def mean_corr(P):
    """나무 두 그루씩 짝지은 예측(행)의 상관계수 평균"""
    C = np.corrcoef(P)
    return C[np.triu_indices_from(C, k=1)].mean()

for mf in [None, "sqrt"]:           # None = 모든 특성 = 배깅
    rf = RandomForestClassifier(n_estimators=200, max_features=mf,
                                min_samples_leaf=20, oob_score=True,
                                random_state=2026, n_jobs=-1)
    rf.fit(Z_train, y_train)
    P = np.array([t.predict_proba(Z_test)[:, 1]
                  for t in rf.estimators_])      # (200, 3750)
    one = np.mean([roc_auc_score(y_test, p) for p in P])
    oob = roc_auc_score(y_train, rf.oob_decision_function_[:, 1])
    print(f"max_features={mf}: corr {mean_corr(P):.3f}, "
          f"one tree AUC {one:.3f}, OOB AUC {oob:.3f}")

### 셀 4. 표본 밖 점수(OOB)와 5겹 교차검증

In [ ]:
rf_pipe = make_tree_pipe(RandomForestClassifier(
    n_estimators=200, min_samples_leaf=20, oob_score=True,
    random_state=2026, n_jobs=-1))
t0 = time.perf_counter()
rf_pipe.fit(X_train, y_train)
t_oob = time.perf_counter() - t0
rf = rf_pipe.named_steps["model"]
print("oob_score_ (accuracy):", round(rf.oob_score_, 4))
p_oob = rf.oob_decision_function_[:, 1]       # 사람마다 OOB 확률
print("OOB AUC:", round(roc_auc_score(y_train, p_oob), 4),
      f" ({t_oob:.0f} s)")

t0 = time.perf_counter()
cv5 = cross_val_score(rf_pipe, X_train, y_train, cv=skf,
                      scoring="roc_auc")
t_cv5 = time.perf_counter() - t0
print("5-fold CV AUC:", round(cv5.mean(), 4), f" ({t_cv5:.0f} s)")

### 셀 5. 로지스틱 회귀, 나무 하나, 배깅, 랜덤 포레스트

In [ ]:
from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression

models = {
    "logistic (ch 0)": make_pipe(
        LogisticRegression(C=0.1, max_iter=1000)),
    "tree (ch 1)": make_tree_pipe(DecisionTreeClassifier(
        max_depth=6, min_samples_leaf=300, random_state=2026)),
    "bagging": make_tree_pipe(BaggingClassifier(
        estimator=DecisionTreeClassifier(min_samples_leaf=20),
        n_estimators=200, random_state=2026, n_jobs=-1)),
    "random forest": make_tree_pipe(RandomForestClassifier(
        n_estimators=200, min_samples_leaf=20,
        random_state=2026, n_jobs=-1))}
rows, fitted = [], {}
for name, m in models.items():
    cv = cross_val_score(m, X_train, y_train, cv=skf,
                         scoring="roc_auc")
    fitted[name] = m.fit(X_train, y_train)   # 학습 자료 전체로
    p = fitted[name].predict_proba(X_test)[:, 1]
    rows.append([name, cv.mean(), cv.std(), roc_auc_score(y_test, p)])
pd.DataFrame(rows, columns=["model", "CV AUC", "CV SD", "test AUC"]
             ).round(3)

## 나. 랜덤 포레스트의 조절값

### 셀 6. 나무 수를 늘리며 OOB AUC 보기 (warm_start)

In [ ]:
forest = make_tree_pipe(RandomForestClassifier(
    min_samples_leaf=20, oob_score=True, warm_start=True,
    random_state=2026, n_jobs=-1))
sizes = [25, 50, 75, 100, 150, 200, 300, 400, 500]
oob_auc = []
t0 = time.perf_counter()
for k in sizes:
    forest.set_params(model__n_estimators=k)  # 나무 수만 늘린다
    forest.fit(X_train, y_train)              # 늘어난 만큼만 새로 키움
    p = forest.named_steps["model"].oob_decision_function_[:, 1]
    oob_auc.append(roc_auc_score(y_train, p))
t_warm = time.perf_counter() - t0
print(np.round(oob_auc, 4), f" ({t_warm:.0f} s)")

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(sizes, oob_auc, marker="o")
ax.set_xlabel("Number of trees (n_estimators)")
ax.set_ylabel("OOB AUC")
plt.tight_layout()

### 셀 7. max_features와 min_samples_leaf의 OOB AUC 열지도

In [ ]:
mf_list = [2, 4, 6, 12, 24, 48]       # 6 = "sqrt", 48 = 배깅
leaf_list = [1, 5, 20, 50, 100]
heat = pd.DataFrame(index=leaf_list, columns=mf_list, dtype=float)
for leaf in leaf_list:
    for mf in mf_list:
        m = make_tree_pipe(RandomForestClassifier(
            n_estimators=200, max_features=mf, min_samples_leaf=leaf,
            oob_score=True, random_state=2026, n_jobs=-1))
        m.fit(X_train, y_train)
        p = m.named_steps["model"].oob_decision_function_[:, 1]
        heat.loc[leaf, mf] = roc_auc_score(y_train, p)

fig, ax = plt.subplots(figsize=(6.2, 3.6))
im = ax.imshow(heat.to_numpy(), cmap="viridis")
for i in range(len(leaf_list)):
    for j in range(len(mf_list)):
        v = heat.iloc[i, j]
        ax.text(j, i, f"{v:.3f}", ha="center", va="center",
                fontsize=9, color="black" if v > 0.74 else "white")
ax.set_xticks(range(len(mf_list)), mf_list)
ax.set_yticks(range(len(leaf_list)), leaf_list)
ax.set_xlabel("max_features")
ax.set_ylabel("min_samples_leaf")
ax.set_title("OOB AUC, 200 trees")
fig.colorbar(im)
heat.round(3)

## 다. 무작위 탐색과 단계적 탐색

### 셀 8. 분포에서 조절값 뽑아 보기 (randint, uniform, loguniform)

In [ ]:
from scipy.stats import randint, uniform, loguniform

print(randint(2, 25).rvs(8, random_state=0))       # 2–24의 정수
print(uniform(0.3, 0.7).rvs(5, random_state=0).round(2))  # 0.3–1.0
print(loguniform(0.001, 100).rvs(5, random_state=0).round(4))

### 셀 9. 격자 탐색 16개 조합 (비교의 기준)

In [ ]:
from sklearn.model_selection import GridSearchCV

skf3 = StratifiedKFold(n_splits=3, shuffle=True, random_state=2026)
rf_base = make_tree_pipe(RandomForestClassifier(
    n_estimators=200, random_state=2026, n_jobs=-1))
grid = {"model__max_features": [3, 6, 12, 24],
        "model__min_samples_leaf": [5, 20, 50, 100]}
gs = GridSearchCV(rf_base, grid, cv=skf3, scoring="roc_auc")
t0 = time.perf_counter()
gs.fit(X_train, y_train)
t_grid = time.perf_counter() - t0
print(gs.best_params_, round(gs.best_score_, 4))
print("candidates: 16, fits: 16 x 3 =", 16 * 3, f" ({t_grid:.0f} s)")

### 셀 10. 무작위 탐색 40개 조합 (RandomizedSearchCV)

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

space = {"model__max_features": randint(2, 25),
         "model__min_samples_leaf": randint(1, 101),
         "model__max_samples": uniform(0.3, 0.7),
         "model__criterion": ["gini", "entropy"]}
rs = RandomizedSearchCV(rf_base, space, n_iter=40, cv=skf3,
                        scoring="roc_auc", random_state=2026)
t0 = time.perf_counter()
rs.fit(X_train, y_train)
t_rand = time.perf_counter() - t0
print(rs.best_params_)
print(round(rs.best_score_, 4), " fits: 40 x 3 =", 40 * 3,
      f" ({t_rand:.0f} s)")

### 셀 11. 무작위 탐색 결과를 표로 (cv_results_)

In [ ]:
res = pd.DataFrame(rs.cv_results_)
show = res[["param_model__max_features",
            "param_model__min_samples_leaf",
            "param_model__max_samples", "param_model__criterion",
            "mean_test_score", "std_test_score", "rank_test_score"]]
show.columns = ["max_feat", "min_leaf", "max_samp", "criterion",
                "mean AUC", "SD", "rank"]
print("best of the first 16 draws:",
      round(res["mean_test_score"][:16].max(), 4))
show.sort_values("rank").head(6).round(3)

### 셀 12. 단계적 탐색 (HalvingRandomSearchCV)

In [ ]:
from sklearn.experimental import enable_halving_search_cv  # noqa
from sklearn.model_selection import HalvingRandomSearchCV

hs = HalvingRandomSearchCV(
    rf_base, space, n_candidates=81, factor=3,
    resource="model__n_estimators", min_resources=22,
    max_resources=200, cv=skf3, scoring="roc_auc",
    random_state=2026)
t0 = time.perf_counter()
hs.fit(X_train, y_train)
t_halv = time.perf_counter() - t0
print(hs.best_params_)
print(round(hs.best_score_, 4), f" ({t_halv:.0f} s)")
steps = pd.DataFrame(hs.cv_results_).groupby("iter").agg(
    candidates=("mean_test_score", "size"),
    trees=("n_resources", "first"),
    best_AUC=("mean_test_score", "max"))
steps.round(4)

### 셀 13. 세 방법의 결과와 시간

In [ ]:
hres = pd.DataFrame(hs.cv_results_)
trees_per_fold = [16 * 200, 40 * 200, hres["n_resources"].sum()]
summary = pd.DataFrame({
    "best CV AUC": [gs.best_score_, rs.best_score_, hs.best_score_],
    "candidates": [16, 40, hs.n_candidates_[0]],
    "fits": [16 * 3, 40 * 3, len(hres) * 3],
    "trees grown": [t * 3 for t in trees_per_fold],
    "seconds": np.round([t_grid, t_rand, t_halv]).astype(int)},
    index=["grid", "random", "halving"])
summary.round(4)

## 라. 베이즈 최적화: Optuna와 HyperOpt

Colab에서는 설치 셀부터 실행합니다(세션마다).

### 셀 14. Optuna와 HyperOpt 설치 (Colab은 세션마다)

In [ ]:
!pip install optuna hyperopt

### 셀 15. Optuna로 40번 시도

In [ ]:
import optuna
from optuna.samplers import TPESampler

def objective(trial):
    """조합 하나를 골라 3겹 교차검증 AUC를 돌려준다"""
    model = RandomForestClassifier(
        n_estimators=200, random_state=2026, n_jobs=-1,
        max_features=trial.suggest_int("max_features", 2, 24),
        min_samples_leaf=trial.suggest_int("min_samples_leaf",
                                           1, 100),
        max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
        criterion=trial.suggest_categorical("criterion",
                                            ["gini", "entropy"]))
    auc = cross_val_score(make_tree_pipe(model), X_train, y_train,
                          cv=skf3, scoring="roc_auc")
    return auc.mean()

optuna.logging.set_verbosity(optuna.logging.WARNING)  # 시도별 로그 끄기
study = optuna.create_study(direction="maximize",
                            sampler=TPESampler(seed=2026))
t0 = time.perf_counter()
study.optimize(objective, n_trials=40)
t_optuna = time.perf_counter() - t0
print(study.best_params)
print(round(study.best_value, 4), " trial", study.best_trial.number,
      f" ({t_optuna:.0f} s)")

### 셀 16. 시도 기록을 표로 (trials_dataframe)

In [ ]:
tdf = study.trials_dataframe(attrs=("number", "value", "params"))
print(tdf.shape)
tdf.sort_values("value", ascending=False).head(5).round(4)

### 셀 17. Optuna 그림 세 가지 (이력, 조절값 중요도, 조각)

In [ ]:
from optuna.visualization.matplotlib import (
    plot_optimization_history, plot_param_importances, plot_slice)
from optuna.importance import FanovaImportanceEvaluator

ax1 = plot_optimization_history(study)
ax2 = plot_param_importances(
    study, evaluator=FanovaImportanceEvaluator(seed=2026))
ax3 = plot_slice(study)

### 셀 18. HyperOpt로 40번 시도

In [ ]:
from hyperopt import fmin, tpe, hp, Trials, STATUS_OK, space_eval

hp_space = {
    "max_features": hp.quniform("max_features", 2, 24, 1),
    "min_samples_leaf": hp.quniform("min_samples_leaf", 1, 100, 1),
    "max_samples": hp.uniform("max_samples", 0.3, 1.0),
    "criterion": hp.choice("criterion", ["gini", "entropy"])}

def hp_objective(params):
    """조합 하나를 받아 손실(-AUC)을 돌려준다"""
    model = RandomForestClassifier(
        n_estimators=200, random_state=2026, n_jobs=-1,
        max_features=int(params["max_features"]),      # 12.0 → 12
        min_samples_leaf=int(params["min_samples_leaf"]),
        max_samples=params["max_samples"],
        criterion=params["criterion"])
    auc = cross_val_score(make_tree_pipe(model), X_train, y_train,
                          cv=skf3, scoring="roc_auc").mean()
    return {"loss": -auc, "status": STATUS_OK}   # 작을수록 좋게

trials = Trials()
t0 = time.perf_counter()
best = fmin(hp_objective, hp_space, algo=tpe.suggest, max_evals=40,
            trials=trials, rstate=np.random.default_rng(2026))
t_hyperopt = time.perf_counter() - t0
print(best)                          # hp.choice는 번호로 나온다
print(space_eval(hp_space, best))    # 실제 값으로 바꾼 것
print(round(-min(trials.losses()), 4), f" ({t_hyperopt:.0f} s)")

### 셀 19. HyperOpt의 Trials를 표와 그림으로

In [ ]:
hdf = pd.DataFrame(trials.vals)        # 시도마다 뽑힌 값
hdf["criterion"] = hdf["criterion"].map({0: "gini", 1: "entropy"})
hdf["AUC"] = [-r["loss"] for r in trials.results]
print(hdf.shape)

fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(hdf.index, hdf["AUC"], "o", alpha=0.6, label="each trial")
ax.plot(hdf.index, hdf["AUC"].cummax(), label="best so far")
ax.set_xlabel("Trial")
ax.set_ylabel("CV AUC (3-fold)")
ax.legend()
plt.tight_layout()
hdf.sort_values("AUC", ascending=False).head(5).round(4)

### 셀 20. 무작위 탐색, Optuna, HyperOpt 비교 (같은 범위, 40번씩)

In [ ]:
so_far = pd.DataFrame({
    "random": res["mean_test_score"].cummax().to_numpy(),
    "Optuna": tdf["value"].cummax().to_numpy(),
    "HyperOpt": hdf["AUC"].cummax().to_numpy()},
    index=range(1, 41))
ax = so_far.plot(figsize=(6.5, 3.2), drawstyle="steps-post")
ax.set_xlabel("Trial")
ax.set_ylabel("Best CV AUC so far")
plt.tight_layout()

pd.DataFrame({"best CV AUC": so_far.iloc[-1],
              "first reached at trial": so_far.idxmax(),
              "seconds": np.round([t_rand, t_optuna,
                                   t_hyperopt]).astype(int)}).round(4)

## 마. 변수 중요도와 결과 정리

### 셀 21. 최종 모형: 학습 자료 전체로 맞추고 시험 자료로 한 번 평가

In [ ]:
import joblib

best_rf = study.best_params                  # 라 절에서 Optuna가 찾은 조합
final = make_tree_pipe(RandomForestClassifier(
    n_estimators=500, random_state=2026, n_jobs=-1, **best_rf))
final.fit(X_train, y_train)                  # 학습 자료 전체로 다시 맞춤
p_final = final.predict_proba(X_test)[:, 1]  # 시험 자료는 여기서 한 번

def boot_ci(y_true, p, n_boot=1000, seed=2026):
    """시험 자료를 복원추출해 AUC의 95% 백분위수 구간"""
    rng = np.random.default_rng(seed)
    yt, out = np.asarray(y_true), []
    for _ in range(n_boot):
        i = rng.integers(0, len(yt), len(yt))
        out.append(roc_auc_score(yt[i], p[i]))
    return np.percentile(out, [2.5, 97.5])

lo, hi = boot_ci(y_test, p_final)
print(f"test AUC {roc_auc_score(y_test, p_final):.3f}"
      f" (95% CI {lo:.3f} to {hi:.3f})")
joblib.dump(final, "rf_final.joblib")        # 저장 (1장 마 절과 같음)

### 셀 22. 불순도 기반 중요도 (feature_importances_)

In [ ]:
names = final[:-1].get_feature_names_out()   # 전처리 뒤 48열의 이름
print(names[:3], names[-3:])

def original(name):
    """원-핫 열 이름을 원래 열로: 'region_metro' → 'region'"""
    for c in cat_cols:
        if name.startswith(c + "_"):
            return c
    return name

mdi = pd.Series(final[-1].feature_importances_, index=names)
mdi = mdi.groupby([original(nm) for nm in names]).sum()  # 39개로
print(len(mdi), round(mdi.sum(), 3))
mdi.sort_values(ascending=False).head(8).round(3)

### 셀 23. 순열 중요도와 나란히 보기 (permutation_importance)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(final, X_test, y_test,
                              scoring="roc_auc", n_repeats=10,
                              random_state=2026)
imp = pd.DataFrame({"impurity": mdi,
                    "perm": pd.Series(perm.importances_mean,
                                      index=X_test.columns),
                    "perm_sd": pd.Series(perm.importances_std,
                                         index=X_test.columns)})
imp["rank_imp"] = imp["impurity"].rank(ascending=False).astype(int)
imp["rank_perm"] = imp["perm"].rank(ascending=False).astype(int)

fig, ax = plt.subplots(1, 2, figsize=(10, 4.6))
a = imp.sort_values("impurity").tail(15)
ax[0].barh(a.index, a["impurity"])
ax[0].set_title("Impurity-based (training data)")
b = imp.sort_values("perm").tail(15)
ax[1].barh(b.index, b["perm"], xerr=b["perm_sd"])
ax[1].set_title("Permutation (test data, AUC drop)")
plt.tight_layout()
check = ["age", "n_drugs", "n_ed", "bmi", "hf", "ckd", "insulin",
         "dyslip", "oa", "region", "n_prescribers"]
imp.loc[check, ["rank_imp", "rank_perm", "perm"]].round(4)

### 셀 24. 서로 상관된 특성은 중요도를 나눠 갖는다

In [ ]:
util = ["n_outpt", "n_drugs", "n_prescribers", "cost_2022"]
print(X_train[util].corr(method="spearman").round(2))

rng = np.random.default_rng(2026)
base = roc_auc_score(y_test, p_final)
drops = []
for _ in range(10):
    Xs = X_test.copy()
    order = rng.permutation(len(Xs))       # 네 열을 같은 순서로 섞기
    Xs[util] = X_test[util].to_numpy()[order]
    drops.append(base - roc_auc_score(
        y_test, final.predict_proba(Xs)[:, 1]))
print("alone   :", imp.loc[util, "perm"].round(4).tolist())
print("together:", round(np.mean(drops), 4))

### 셀 25. 아무 의미 없는 열을 더해 보면

In [ ]:
rng = np.random.default_rng(7)
Xn_train = X_train.assign(noise_num=rng.normal(size=len(X_train)),
                          noise_bin=rng.integers(0, 2, len(X_train)))
Xn_test = X_test.assign(noise_num=rng.normal(size=len(X_test)),
                        noise_bin=rng.integers(0, 2, len(X_test)))
noisy = make_tree_pipe(RandomForestClassifier(
    n_estimators=200, random_state=2026, n_jobs=-1, **best_rf),
    num=num_cols + ["noise_num", "noise_bin"])
noisy.fit(Xn_train, y_train)

nm = noisy[:-1].get_feature_names_out()
mdi_n = pd.Series(noisy[-1].feature_importances_, index=nm)
mdi_n = mdi_n.groupby([original(x) for x in nm]).sum()
perm_n = permutation_importance(noisy, Xn_test, y_test,
                                scoring="roc_auc", n_repeats=10,
                                random_state=2026)
out = pd.DataFrame({
    "impurity": mdi_n,
    "rank_imp": mdi_n.rank(ascending=False).astype(int),
    "perm": pd.Series(perm_n.importances_mean,
                      index=Xn_test.columns)})
out.loc[["noise_num", "noise_bin", "hf", "insulin"]].round(4)

### 셀 26. 부분 의존 그림 (PartialDependenceDisplay)

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

screened = X_train[X_train["bmi"].notna()]    # BMI가 있는 사람만
X_pdp = screened.sample(1000, random_state=2026)
X_pdp = X_pdp.astype({c: float for c in num_cols})  # 정수 열을 실수로
fig, ax = plt.subplots(1, 4, figsize=(12, 3.2), sharey=True)
pdp = PartialDependenceDisplay.from_estimator(
    final, X_pdp, ["age", "bmi", "n_ed", "n_drugs"], ax=ax,
    percentiles=(0.01, 0.99), grid_resolution=40)
plt.tight_layout()

### 셀 27. 모형 비교 표와 ROC 곡선

In [ ]:
from sklearn.metrics import roc_curve

p_test = {name: m.predict_proba(X_test)[:, 1]
          for name, m in fitted.items()}       # 셀 5에서 맞춘 네 모형
p_test["tuned forest"] = p_final
cv_final = cross_val_score(final, X_train, y_train, cv=skf,
                           scoring="roc_auc")
cv_auc = [r[1] for r in rows] + [cv_final.mean()]

table, fig = [], plt.figure(figsize=(5.2, 4.8))
for (name, p), cv in zip(p_test.items(), cv_auc):
    auc = roc_auc_score(y_test, p)
    lo, hi = boot_ci(y_test, p)
    table.append([name, cv, auc, lo, hi])
    fpr, tpr, _ = roc_curve(y_test, p)
    plt.plot(fpr, tpr, label=f"{name} ({auc:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray")
plt.xlabel("1 - specificity")
plt.ylabel("Sensitivity")
plt.legend(fontsize=8)
pd.DataFrame(table, columns=["model", "CV AUC", "test AUC",
                             "CI low", "CI high"]).round(3)

## 과제 정답

**과제 1.** class_weight="balanced_subsample"로 Optuna 탐색을 다시 하고(20번), 기본 모형과 시험 AUC, 평균 예측 확률, 임계값 0.10의 혼동행렬을 비교합니다.

### 셀 28. 과제 1 정답. class_weight를 바꿔 다시 튜닝

In [ ]:
from sklearn.metrics import confusion_matrix

def objective_bal(trial):
    model = RandomForestClassifier(
        n_estimators=200, random_state=2026, n_jobs=-1,
        class_weight="balanced_subsample",
        max_features=trial.suggest_int("max_features", 2, 24),
        min_samples_leaf=trial.suggest_int("min_samples_leaf",
                                           1, 100),
        max_samples=trial.suggest_float("max_samples", 0.3, 1.0),
        criterion=trial.suggest_categorical("criterion",
                                            ["gini", "entropy"]))
    return cross_val_score(make_tree_pipe(model), X_train, y_train,
                           cv=skf3, scoring="roc_auc").mean()

study_bal = optuna.create_study(direction="maximize",
                                sampler=TPESampler(seed=2026))
study_bal.optimize(objective_bal, n_trials=20)
print(study_bal.best_params, round(study_bal.best_value, 4))

bal = make_tree_pipe(RandomForestClassifier(
    n_estimators=500, random_state=2026, n_jobs=-1,
    class_weight="balanced_subsample", **study_bal.best_params))
bal.fit(X_train, y_train)
p_bal = bal.predict_proba(X_test)[:, 1]
for name, p in [("default", p_final), ("balanced_subsample", p_bal)]:
    print(f"{name}: test AUC {roc_auc_score(y_test, p):.3f},"
          f" mean prob {p.mean():.3f}")
    print(confusion_matrix(y_test, (p >= 0.10).astype(int)))

**과제 2.** 셀 15의 study를 40번 더 이어서 돌려 시도 10, 20, 40, 80번째까지의 최고 점수를 봅니다.

### 셀 29. 과제 2 정답. 시도 수에 따른 최고 점수

In [ ]:
study.optimize(objective, n_trials=40)       # 40번을 더 (모두 80번)
best_by = study.trials_dataframe()["value"].cummax()
for n_t in [10, 20, 40, 80]:
    print(f"after {n_t:2d} trials:",
          f"best CV AUC {best_by[n_t - 1]:.4f}")
print(study.best_params)

**과제 3.** 최종 조합으로 나무 수를 10–1000으로 바꿔 시험 AUC와 걸린 시간을 봅니다.

### 셀 30. 과제 3 정답. 나무 수에 따른 시험 AUC와 시간

In [ ]:
out = []
for k in [10, 50, 100, 200, 500, 1000]:
    m = make_tree_pipe(RandomForestClassifier(
        n_estimators=k, random_state=2026, n_jobs=-1, **best_rf))
    t0 = time.perf_counter()
    m.fit(X_train, y_train)
    sec = time.perf_counter() - t0
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    out.append([k, round(auc, 4), round(sec, 1)])
pd.DataFrame(out, columns=["n_estimators", "test AUC", "seconds"])